# 05b - Chatbot (Module 3) report

Scores in `05_chat/report.py` (definitions in its docstring). Writes `results/chat/chat.json`, `chat_rows.csv` and
`results/figures/fig_chat.png`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "PLAN.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for p in (ROOT, ROOT / "05_chat", ROOT / "03_rag"):
    sys.path.insert(0, str(p))

import report
from common import config, env

df, summary = report.build()
report.OUT.mkdir(parents=True, exist_ok=True)
df.drop(columns=["reply"]).to_csv(report.OUT / "chat_rows.csv", index=False)
env.write_json(report.OUT / "chat.json", summary)
import pandas as pd

pd.DataFrame({s: {k: round(v["mean"], 3) for k, v in m.items() if isinstance(v, dict)}
              for s, m in summary["systems"].items()})

In [ ]:
pd.DataFrame({s: {g: summary["by_group"][s][g].get("task_ok", {}).get("mean") for g in report.GROUPS}
              for s in ("PQ", "SK")}).round(2)

In [ ]:
print("Wilcoxon PQ vs SK:", {k: f"{v:.2g}" for k, v in summary["wilcoxon_pq_vs_sk"].items()})
print("pipeline latency without the LLM (s):", summary["latency_without_llm_s"])
print("weather keyword false positives:", summary["weather_false_positives"])
print("memory:", summary["memory"])
df[(df.system == "PQ") & (df.correct < 1)][["case", "correct", "note"]]

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

groups = report.GROUPS
fig, ax = plt.subplots(figsize=(6, 2.6))
x = np.arange(len(groups))
for i, (s, name) in enumerate((("PQ", "PlantIQ chatbot"), ("SK", "Selva Kumar et al. (LangChain)"))):
    v = [summary["by_group"][s][g]["task_ok"]["mean"] for g in groups]
    ax.bar(x - 0.2 + 0.4 * i, v, 0.38, label=name)
ax.set_xticks(x, [f"{g} ({summary['by_group']['PQ'][g]['n']})" for g in groups])
ax.set_ylabel("Task success"); ax.set_ylim(0, 1.05); ax.legend(frameon=False)
config.FIG_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(config.FIG_DIR / "fig_chat.png", dpi=200, bbox_inches="tight")
plt.show()